# Polynomial Regression

This notebook builds on linear regression to handle **non-linear relationships** by transforming features into polynomial terms.

## Key Concepts
- **Polynomial Features**: Transform `x` into `[1, x, x², x³, ...]`
- **Degree Selection**: Balancing underfitting vs overfitting
- **Regularization**: Preventing overfitting with Ridge regression

## Clinical Example: Drug Dose vs Plasma Concentration
For most drugs, doubling the dose doubles the blood concentration. For a drug with
**saturable (capacity-limited) metabolism** it does not, because the liver enzyme that
clears the drug has a maximum throughput, `Vmax`:

- At low doses the enzyme has spare capacity. Clearance is first-order and concentration rises roughly in step with dose.
- As the dose approaches `Vmax` the enzyme **saturates**. Each additional milligram is cleared more slowly than the last, so concentration rises **faster than dose** — the curve bends sharply upward.

This is the Michaelis-Menten relationship `Css = Km · Dose / (Vmax − Dose)`, and it is
exactly why drugs like phenytoin need careful dose titration: a small dose increase near
saturation can push a patient from therapeutic to toxic.

A straight line cannot capture that curve — which is the whole point of this notebook.

> **A note on the data.** Every measurement here is simulated from a pharmacokinetic
> model. No real or identifiable patient data is used.


In [1]:
import numpy as np
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from IPython.display import display, clear_output
import time

---
## 1. Why Linear Regression Fails

Let's see what happens when we try to fit a straight line to the dose-concentration data.

**Run the cell twice** to toggle between:
- Initial state (no fit)
- Linear fit (clearly inadequate)


In [2]:
# Why Linear Regression Fails - Clinical Example: Concentration vs Dose
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- TOGGLE LOGIC ---
try:
    show_linear_fit
except NameError:
    show_linear_fit = False

show_linear_fit = not show_linear_fit

# --- 1. GENERATE NON-LINEAR DATA ---
# Simulated pharmacokinetic study - no real patient data is used.
np.random.seed(42)
n_points = 50
# Daily dose of an orally administered drug (5 to 25 mg/day)
dose = np.linspace(5, 25, n_points)

# Steady-state plasma concentration under SATURABLE (capacity-limited) elimination.
# Michaelis-Menten:  Css = Km * Dose / (Vmax - Dose)
#   Vmax = 30 mg/day is the most the metabolizing enzyme can clear in a day
#   Km   = 4  mg/L   is the concentration at half-maximal clearance
# Well below Vmax the enzyme has spare capacity and concentration rises almost
# in step with dose. As the dose approaches Vmax the enzyme saturates, each extra
# milligram is cleared more slowly, and concentration rises FASTER than dose.
Vmax, Km = 30.0, 4.0
true_conc = Km * dose / (Vmax - dose)
conc = true_conc + np.random.normal(0, 1.2, n_points)
conc = np.maximum(conc, 0.2)  # Concentration can't be negative

# --- 2. FIT LINEAR MODEL ---
if show_linear_fit:
    # Simple linear regression: y = mx + b
    X_linear = np.column_stack([np.ones(n_points), dose])
    theta = np.linalg.lstsq(X_linear, conc, rcond=None)[0]
    linear_pred = X_linear @ theta

    # Calculate R² for linear fit
    ss_res = np.sum((conc - linear_pred)**2)
    ss_tot = np.sum((conc - np.mean(conc))**2)
    r2_linear = 1 - ss_res / ss_tot

    status = f"Linear Fit | R² = {r2_linear:.3f} (Poor fit!)"
    line_color = 'red'
else:
    linear_pred = np.zeros_like(dose)
    status = "Initial State - Run again to see linear fit"
    line_color = 'gray'

# --- 3. VISUALIZE ---
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.65, 0.35],
    subplot_titles=('Plasma Concentration vs Daily Dose', 'Residuals')
)

# Data points
fig.add_trace(go.Scatter(
    x=dose, y=conc,
    mode='markers',
    marker=dict(size=8, color='blue', opacity=0.6),
    name='Observed'
), row=1, col=1)

# True relationship (dashed)
fig.add_trace(go.Scatter(
    x=dose, y=true_conc,
    mode='lines',
    line=dict(color='green', width=2, dash='dash'),
    name='True (saturable PK)'
), row=1, col=1)

# Linear fit
if show_linear_fit:
    fig.add_trace(go.Scatter(
        x=dose, y=linear_pred,
        mode='lines',
        line=dict(color=line_color, width=3),
        name='Linear fit'
    ), row=1, col=1)

    # Residuals
    residuals = conc - linear_pred
    fig.add_trace(go.Scatter(
        x=dose, y=residuals,
        mode='markers',
        marker=dict(size=6, color='red', opacity=0.6),
        name='Residuals'
    ), row=1, col=2)

    # Zero line
    fig.add_hline(y=0, line_dash="dash", line_color="gray", row=1, col=2)

fig.update_xaxes(title_text="Dose (mg/day)", row=1, col=1)
fig.update_yaxes(title_text="Plasma Concentration (mg/L)", row=1, col=1)
fig.update_xaxes(title_text="Dose (mg/day)", row=1, col=2)
fig.update_yaxes(title_text="Residual", row=1, col=2)

fig.update_layout(
    title=status,
    width=1000, height=450,
    showlegend=True,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

if show_linear_fit:
    print("Notice: Residuals show a clear PATTERN (curved), indicating the linear model")
    print("is missing systematic structure in the data. A good fit has random residuals.")
    print("Clinically: the straight line over-predicts concentration at low doses and")
    print("badly under-predicts it near saturation - the doses where toxicity happens.")


Notice: Residuals show a clear PATTERN (curved), indicating the linear model
is missing systematic structure in the data. A good fit has random residuals.
Clinically: the straight line over-predicts concentration at low doses and
badly under-predicts it near saturation - the doses where toxicity happens.


---
## 2. Polynomial Feature Transformation

The key insight: **polynomial regression is still linear regression**, just with transformed features!

For a single feature `x`, we create:
- Degree 1: `[1, x]`
- Degree 2: `[1, x, x²]`
- Degree 3: `[1, x, x², x³]`

Then we fit: `y = θ₀ + θ₁x + θ₂x² + θ₃x³ + ...`

**Run the cell twice** to toggle between showing the transformation.

In [3]:
# Polynomial Feature Transformation Visualization
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- TOGGLE LOGIC ---
try:
    show_poly_transform
except NameError:
    show_poly_transform = False

show_poly_transform = not show_poly_transform

# --- EXAMPLE DATA ---
x_example = np.array([5, 10, 15, 20, 25])

# --- CREATE POLYNOMIAL FEATURES ---
degree = 3
X_poly = np.column_stack([x_example**i for i in range(degree + 1)])

# --- VISUALIZE ---
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.4, 0.6],
    subplot_titles=('Original Feature', 'Polynomial Features (Degree 3)')
)

# Original feature
fig.add_trace(go.Bar(
    x=[f'x={v}' for v in x_example],
    y=x_example,
    marker_color='blue',
    name='x'
), row=1, col=1)

if show_poly_transform:
    # Polynomial features (normalized for visualization)
    colors = ['lightgray', 'blue', 'green', 'red']
    labels = ['1 (bias)', 'x', 'x²', 'x³']
    
    # Show each polynomial term
    for i, (label, color) in enumerate(zip(labels, colors)):
        # Normalize for visualization
        values = X_poly[:, i]
        if i > 0:
            values = values / values.max() * 100  # Scale to 0-100
        else:
            values = np.ones(len(x_example)) * 10  # Bias term
        
        fig.add_trace(go.Bar(
            x=[f'x={v}' for v in x_example],
            y=values,
            name=label,
            marker_color=color,
            opacity=0.7
        ), row=1, col=2)
    
    status = "Polynomial Features Created"
    
    # Print actual values
    print("Polynomial Feature Matrix (degree=3):")
    print(f"{'x':>6} | {'1':>6} | {'x':>8} | {'x²':>10} | {'x³':>12}")
    print("-" * 50)
    for i, x in enumerate(x_example):
        print(f"{x:>6} | {1:>6} | {x:>8} | {x**2:>10} | {x**3:>12}")
else:
    status = "Initial - Run again to see transformation"

fig.update_layout(
    title=status,
    width=1000, height=400,
    barmode='group',
    showlegend=True
)

fig.update_yaxes(title_text="Value", row=1, col=1)
fig.update_yaxes(title_text="Scaled Value", row=1, col=2)

fig.show()

Polynomial Feature Matrix (degree=3):
     x |      1 |        x |         x² |           x³
--------------------------------------------------
     5 |      1 |        5 |         25 |          125
    10 |      1 |       10 |        100 |         1000
    15 |      1 |       15 |        225 |         3375
    20 |      1 |       20 |        400 |         8000
    25 |      1 |       25 |        625 |        15625


---
## 3. Fitting Polynomial Regression

Now let's fit polynomial models of different degrees to our dose-concentration data.

**Run the cell twice** to toggle between:
- Initial state
- Cubic polynomial fit (degree 3)


In [4]:
# Polynomial Regression Fit - Clinical Example: Concentration vs Dose
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- TOGGLE LOGIC ---
try:
    show_poly_fit
except NameError:
    show_poly_fit = False

show_poly_fit = not show_poly_fit

# --- 1. DATA ---
# Simulated pharmacokinetic study - no real patient data is used.
np.random.seed(42)
n_points = 50
# Daily dose of an orally administered drug (5 to 25 mg/day)
dose = np.linspace(5, 25, n_points)

# Steady-state plasma concentration under SATURABLE (capacity-limited) elimination.
# Michaelis-Menten:  Css = Km * Dose / (Vmax - Dose)
#   Vmax = 30 mg/day is the most the metabolizing enzyme can clear in a day
#   Km   = 4  mg/L   is the concentration at half-maximal clearance
# Well below Vmax the enzyme has spare capacity and concentration rises almost
# in step with dose. As the dose approaches Vmax the enzyme saturates, each extra
# milligram is cleared more slowly, and concentration rises FASTER than dose.
Vmax, Km = 30.0, 4.0
true_conc = Km * dose / (Vmax - dose)
conc = true_conc + np.random.normal(0, 1.2, n_points)
conc = np.maximum(conc, 0.2)  # Concentration can't be negative

# --- 2. FIT POLYNOMIAL (DEGREE 3) ---
degree = 3

if show_poly_fit:
    # Create polynomial features
    X_poly = np.column_stack([dose**i for i in range(degree + 1)])

    # Solve using normal equation
    theta = np.linalg.lstsq(X_poly, conc, rcond=None)[0]

    # Predictions (smooth curve)
    dose_smooth = np.linspace(5, 25, 200)
    X_smooth = np.column_stack([dose_smooth**i for i in range(degree + 1)])
    poly_pred = X_smooth @ theta

    # R² calculation
    pred_train = X_poly @ theta
    ss_res = np.sum((conc - pred_train)**2)
    ss_tot = np.sum((conc - np.mean(conc))**2)
    r2 = 1 - ss_res / ss_tot

    status = f"Polynomial Fit (Degree {degree}) | R² = {r2:.4f}"
    coef_colors = ['green' if c >= 0 else 'red' for c in theta]
else:
    theta = np.zeros(degree + 1)
    dose_smooth = np.linspace(5, 25, 200)
    poly_pred = np.zeros_like(dose_smooth)
    status = "Initial - Run again to fit polynomial"
    coef_colors = ['gray'] * (degree + 1)

# --- 3. VISUALIZE ---
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=('Plasma Concentration vs Dose', 'Coefficients')
)

# Data points
fig.add_trace(go.Scatter(
    x=dose, y=conc,
    mode='markers',
    marker=dict(size=8, color='blue', opacity=0.6),
    name='Observed'
), row=1, col=1)

# True relationship
fig.add_trace(go.Scatter(
    x=dose_smooth, y=Km * dose_smooth / (Vmax - dose_smooth),
    mode='lines',
    line=dict(color='green', width=2, dash='dash'),
    name='True (Michaelis-Menten)'
), row=1, col=1)

# Polynomial fit
if show_poly_fit:
    fig.add_trace(go.Scatter(
        x=dose_smooth, y=poly_pred,
        mode='lines',
        line=dict(color='red', width=3),
        name=f'Poly fit (deg={degree})'
    ), row=1, col=1)

# Coefficients
coef_labels = ['θ₀ (bias)', 'θ₁ (x)', 'θ₂ (x²)', 'θ₃ (x³)']
fig.add_trace(go.Bar(
    x=coef_labels,
    y=theta,
    marker_color=coef_colors,
    text=[f'{c:.4f}' for c in theta],
    textposition='outside',
    name='Coefficients'
), row=1, col=2)

fig.update_xaxes(title_text="Dose (mg/day)", row=1, col=1)
fig.update_yaxes(title_text="Plasma Concentration (mg/L)", row=1, col=1)
fig.update_yaxes(title_text="Value", row=1, col=2)

fig.update_layout(
    title=status,
    width=1000, height=450,
    showlegend=True,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

if show_poly_fit:
    print(f"\nFitted equation: conc = {theta[0]:.3f} + {theta[1]:.4f}×dose + {theta[2]:.4f}×dose² + {theta[3]:.6f}×dose³")
    print(f"True mechanism:  conc = {Km:.0f}×dose / ({Vmax:.0f} − dose)   (saturable elimination)")
    print("\nThe cubic is an APPROXIMATION of the pharmacology, not the pharmacology itself.")
    print("It tracks the curve well inside the observed dose range, but it has no Vmax,")
    print("so do not trust it to extrapolate beyond 25 mg/day.")



Fitted equation: conc = -0.886 + 0.7577×dose + -0.0807×dose² + 0.003236×dose³
True mechanism:  conc = 4×dose / (30 − dose)   (saturable elimination)

The cubic is an APPROXIMATION of the pharmacology, not the pharmacology itself.
It tracks the curve well inside the observed dose range, but it has no Vmax,
so do not trust it to extrapolate beyond 25 mg/day.


---
## 4. Animated Polynomial Fitting

Watch gradient descent optimize the polynomial coefficients in real-time.

**Run the cell twice** to toggle between:
- Reset to zero
- Animated training

In [5]:
# Animated Polynomial Regression Training
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import clear_output
import time

# --- TOGGLE LOGIC ---
try:
    run_poly_training
except NameError:
    run_poly_training = False

run_poly_training = not run_poly_training

# --- 1. DATA ---
# Simulated pharmacokinetic study - no real patient data is used.
np.random.seed(42)
n_points = 50
# Daily dose of an orally administered drug (5 to 25 mg/day)
dose = np.linspace(5, 25, n_points)

# Steady-state plasma concentration under SATURABLE (capacity-limited) elimination.
# Michaelis-Menten:  Css = Km * Dose / (Vmax - Dose)
#   Vmax = 30 mg/day is the most the metabolizing enzyme can clear in a day
#   Km   = 4  mg/L   is the concentration at half-maximal clearance
# Well below Vmax the enzyme has spare capacity and concentration rises almost
# in step with dose. As the dose approaches Vmax the enzyme saturates, each extra
# milligram is cleared more slowly, and concentration rises FASTER than dose.
Vmax, Km = 30.0, 4.0
true_conc = Km * dose / (Vmax - dose)
conc = true_conc + np.random.normal(0, 1.2, n_points)
conc = np.maximum(conc, 0.2)  # Concentration can't be negative

# Normalize dose for stable training
dose_mean, dose_std = dose.mean(), dose.std()
dose_norm = (dose - dose_mean) / dose_std

# Create polynomial features (degree 3)
degree = 3
X_poly_np = np.column_stack([dose_norm**i for i in range(1, degree + 1)])

# Convert to tensors
X_tensor = torch.from_numpy(X_poly_np.astype(np.float32))
y_tensor = torch.from_numpy(conc.astype(np.float32)).view(-1, 1)

# --- 2. MODEL ---
class PolyModel(nn.Module):
    def __init__(self, degree):
        super().__init__()
        self.linear = nn.Linear(degree, 1)

    def forward(self, x):
        return self.linear(x)

model = PolyModel(degree)

# --- HELPER FUNCTION ---
def create_training_figure(dose, conc, predictions, coeffs, losses, title):
    fig = make_subplots(
        rows=2, cols=2,
        column_widths=[0.7, 0.3],
        row_heights=[0.6, 0.4],
        subplot_titles=('Concentration vs Dose', 'Coefficients', 'Loss'),
        specs=[[{"rowspan": 2}, {}], [None, {}]]
    )

    # Data
    fig.add_trace(go.Scatter(
        x=dose, y=conc, mode='markers',
        marker=dict(size=6, color='blue', opacity=0.5),
        name='Data'
    ), row=1, col=1)

    # Predictions
    sort_idx = np.argsort(dose)
    fig.add_trace(go.Scatter(
        x=dose[sort_idx], y=predictions[sort_idx],
        mode='lines', line=dict(color='red', width=3),
        name='Fit'
    ), row=1, col=1)

    # Coefficients
    coef_names = ['θ₁ (x)', 'θ₂ (x²)', 'θ₃ (x³)', 'bias']
    colors = ['green' if c >= 0 else 'red' for c in coeffs]
    fig.add_trace(go.Bar(
        x=coef_names, y=coeffs,
        marker_color=colors,
        text=[f'{c:.2f}' for c in coeffs],
        textposition='outside'
    ), row=1, col=2)

    # Loss
    fig.add_trace(go.Scatter(
        x=list(range(len(losses))), y=losses,
        mode='lines', line=dict(color='blue')
    ), row=2, col=2)

    fig.update_xaxes(title_text="Dose (mg/day)", row=1, col=1)
    fig.update_yaxes(title_text="Concentration (mg/L)", row=1, col=1)
    fig.update_yaxes(range=[0, 25], row=1, col=1)
    fig.update_yaxes(range=[-2, 8], row=1, col=2)

    fig.update_layout(
        title=title,
        width=1000, height=550,
        showlegend=False
    )

    return fig

# --- EXECUTION ---
if not run_poly_training:
    # Reset
    with torch.no_grad():
        model.linear.weight.fill_(0.0)
        model.linear.bias.fill_(0.0)
        predictions = model(X_tensor).numpy().flatten()

    coeffs = list(model.linear.weight.data.numpy().flatten()) + [model.linear.bias.item()]
    fig = create_training_figure(
        dose, conc, predictions, coeffs, [],
        'State: RESET - Run again to train'
    )
    fig.show()

else:
    # Initialize
    with torch.no_grad():
        model.linear.weight.fill_(0.0)
        model.linear.bias.fill_(0.0)

    optimizer = optim.Adam(model.parameters(), lr=0.05)
    criterion = nn.MSELoss()

    losses = []
    num_epochs = 500

    for epoch in range(num_epochs):
        optimizer.zero_grad()
        outputs = model(X_tensor)
        loss = criterion(outputs, y_tensor)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

        # Update visualization
        update_freq = 1 if epoch < 20 else 10
        if (epoch + 1) % update_freq == 0:
            clear_output(wait=True)

            with torch.no_grad():
                predictions = model(X_tensor).numpy().flatten()

            coeffs = list(model.linear.weight.data.numpy().flatten()) + [model.linear.bias.item()]
            fig = create_training_figure(
                dose, conc, predictions, coeffs, losses,
                f'State: TRAINING | Epoch {epoch+1}/{num_epochs} | Loss: {loss.item():.2f}'
            )
            fig.show()
            time.sleep(0.05)


---
## 5. The Bias-Variance Tradeoff: Degree Selection

This is the most important concept in polynomial regression!

- **Low degree (underfitting)**: Model is too simple, high bias
- **High degree (overfitting)**: Model memorizes noise, high variance
- **Right degree**: Captures the pattern without memorizing noise

**Run the cell multiple times** to cycle through degrees 1 → 3 → 10 → 20

In [6]:
# Degree Selection - Underfitting vs Overfitting
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- TOGGLE LOGIC ---
try:
    degree_idx
except NameError:
    degree_idx = -1

degrees = [1, 3, 10, 20]
degree_idx = (degree_idx + 1) % len(degrees)
current_degree = degrees[degree_idx]

# --- 1. DATA (fewer points to show overfitting) ---
# Simulated pharmacokinetic study - no real patient data is used.
np.random.seed(42)
n_points = 20
# Daily dose of an orally administered drug (5 to 25 mg/day)
dose = np.linspace(5, 25, n_points)

# Steady-state plasma concentration under SATURABLE (capacity-limited) elimination.
# Michaelis-Menten:  Css = Km * Dose / (Vmax - Dose)
#   Vmax = 30 mg/day is the most the metabolizing enzyme can clear in a day
#   Km   = 4  mg/L   is the concentration at half-maximal clearance
# Well below Vmax the enzyme has spare capacity and concentration rises almost
# in step with dose. As the dose approaches Vmax the enzyme saturates, each extra
# milligram is cleared more slowly, and concentration rises FASTER than dose.
Vmax, Km = 30.0, 4.0
true_conc = Km * dose / (Vmax - dose)
conc = true_conc + np.random.normal(0, 2.5, n_points)
conc = np.maximum(conc, 0.2)  # Concentration can't be negative

# --- 2. FIT POLYNOMIAL ---
X_poly = np.column_stack([dose**i for i in range(current_degree + 1)])

# Use ridge regression for numerical stability with high degrees
lambda_reg = 1e-10
theta = np.linalg.solve(
    X_poly.T @ X_poly + lambda_reg * np.eye(current_degree + 1),
    X_poly.T @ conc
)

# Predictions
dose_smooth = np.linspace(5, 25, 500)
X_smooth = np.column_stack([dose_smooth**i for i in range(current_degree + 1)])
pred_smooth = X_smooth @ theta

# Training predictions for R²
pred_train = X_poly @ theta
ss_res = np.sum((conc - pred_train)**2)
ss_tot = np.sum((conc - np.mean(conc))**2)
r2_train = 1 - ss_res / ss_tot

# Determine status
if current_degree == 1:
    status_label = "UNDERFITTING"
    status_color = "orange"
elif current_degree == 3:
    status_label = "GOOD FIT"
    status_color = "green"
else:
    status_label = "OVERFITTING"
    status_color = "red"

# --- 3. VISUALIZE ---
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.7, 0.3],
    subplot_titles=(
        f'Polynomial Degree = {current_degree}',
        'Model Complexity'
    )
)

# Data points
fig.add_trace(go.Scatter(
    x=dose, y=conc,
    mode='markers',
    marker=dict(size=10, color='blue'),
    name='Data'
), row=1, col=1)

# True function
fig.add_trace(go.Scatter(
    x=dose_smooth, y=Km * dose_smooth / (Vmax - dose_smooth),
    mode='lines',
    line=dict(color='green', width=2, dash='dash'),
    name='True function'
), row=1, col=1)

# Fitted polynomial
fig.add_trace(go.Scatter(
    x=dose_smooth, y=pred_smooth,
    mode='lines',
    line=dict(color='red', width=3),
    name=f'Degree {current_degree}'
), row=1, col=1)

# Complexity indicator
fig.add_trace(go.Bar(
    x=['Parameters'],
    y=[current_degree + 1],
    marker_color=status_color,
    text=[f'{current_degree + 1}'],
    textposition='outside',
    name='# Params'
), row=1, col=2)

fig.update_xaxes(title_text="Dose (mg/day)", row=1, col=1)
fig.update_yaxes(title_text="Concentration (mg/L)", range=[-10, 40], row=1, col=1)
fig.update_yaxes(range=[0, 25], row=1, col=2)

fig.update_layout(
    title=f'{status_label} | Degree={current_degree} | R²={r2_train:.4f} | Run again to change degree',
    width=1000, height=450,
    showlegend=True,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

print(f"\nDegree {current_degree}: {current_degree + 1} parameters")
if current_degree == 1:
    print("→ Too simple! A straight line cannot bend with the saturation curve.")
elif current_degree == 3:
    print("→ Just right! Tracks the saturating dose-concentration curve.")
elif current_degree == 10:
    print("→ Starting to overfit. Wiggles appearing between patients.")
else:
    print("→ Severe overfitting! Memorizing assay noise, wild oscillations.")
    print("   A dose-response curve like this would be dangerous to titrate against.")



Degree 1: 2 parameters
→ Too simple! A straight line cannot bend with the saturation curve.


---
## 6. Training vs Validation Error

The key to detecting overfitting: **compare training error with validation error**.

- Training error always decreases with model complexity
- Validation error decreases then **increases** (overfitting!)

**Run the cell twice** to see the analysis.

In [7]:
# Training vs Validation Error
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- TOGGLE LOGIC ---
try:
    show_train_val
except NameError:
    show_train_val = False

show_train_val = not show_train_val

# --- 1. DATA ---
# Simulated pharmacokinetic study - no real patient data is used.
np.random.seed(42)
n_total = 40
dose_all = np.linspace(5, 25, n_total)
# Saturable (Michaelis-Menten) elimination, as in the cells above
Vmax, Km = 30.0, 4.0
conc_all = Km * dose_all / (Vmax - dose_all) + np.random.normal(0, 1.4, n_total)
conc_all = np.maximum(conc_all, 0.2)

# Split into train/validation
indices = np.random.permutation(n_total)
n_train = 25
train_idx = indices[:n_train]
val_idx = indices[n_train:]

dose_train, conc_train = dose_all[train_idx], conc_all[train_idx]
dose_val, conc_val = dose_all[val_idx], conc_all[val_idx]

# --- 2. COMPUTE ERRORS FOR EACH DEGREE ---
if show_train_val:
    max_degree = 15
    degrees = range(1, max_degree + 1)
    train_errors = []
    val_errors = []

    for d in degrees:
        # Create features
        X_train = np.column_stack([dose_train**i for i in range(d + 1)])
        X_val = np.column_stack([dose_val**i for i in range(d + 1)])

        # Fit with small regularization for stability
        lambda_reg = 1e-8
        theta = np.linalg.solve(
            X_train.T @ X_train + lambda_reg * np.eye(d + 1),
            X_train.T @ conc_train
        )

        # Compute MSE
        train_pred = X_train @ theta
        val_pred = X_val @ theta

        train_mse = np.mean((conc_train - train_pred)**2)
        val_mse = np.mean((conc_val - val_pred)**2)

        train_errors.append(train_mse)
        val_errors.append(val_mse)

    # Find best degree
    best_degree = degrees[np.argmin(val_errors)]
    status = f"Analysis Complete | Best Degree = {best_degree}"
else:
    degrees = range(1, 16)
    train_errors = [0] * 15
    val_errors = [0] * 15
    best_degree = 0
    status = "Initial - Run again to compute"

# --- 3. VISUALIZE ---
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.5, 0.5],
    subplot_titles=('Train/Validation MSE vs Degree', 'Data Split')
)

if show_train_val:
    # Error curves
    fig.add_trace(go.Scatter(
        x=list(degrees), y=train_errors,
        mode='lines+markers',
        line=dict(color='blue', width=2),
        marker=dict(size=6),
        name='Training Error'
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=list(degrees), y=val_errors,
        mode='lines+markers',
        line=dict(color='red', width=2),
        marker=dict(size=6),
        name='Validation Error'
    ), row=1, col=1)

    # Best degree marker
    fig.add_vline(
        x=best_degree, line_dash="dash", line_color="green",
        annotation_text=f"Best: {best_degree}", row=1, col=1
    )

# Data points
fig.add_trace(go.Scatter(
    x=dose_train, y=conc_train,
    mode='markers',
    marker=dict(size=8, color='blue', symbol='circle'),
    name='Training Data'
), row=1, col=2)

fig.add_trace(go.Scatter(
    x=dose_val, y=conc_val,
    mode='markers',
    marker=dict(size=8, color='red', symbol='x'),
    name='Validation Data'
), row=1, col=2)

fig.update_xaxes(title_text="Polynomial Degree", row=1, col=1)
fig.update_yaxes(title_text="Mean Squared Error", row=1, col=1)
fig.update_xaxes(title_text="Dose (mg/day)", row=1, col=2)
fig.update_yaxes(title_text="Concentration (mg/L)", row=1, col=2)

if show_train_val:
    fig.update_yaxes(range=[0, max(val_errors) * 1.2], row=1, col=1)

fig.update_layout(
    title=status,
    width=1000, height=400,
    showlegend=True
)

fig.show()

if show_train_val:
    print(f"\nKey insight: Training error keeps decreasing, but validation error")
    print(f"increases after degree {best_degree} - that's OVERFITTING!")
    print(f"\nOptimal degree: {best_degree}")



Key insight: Training error keeps decreasing, but validation error
increases after degree 2 - that's OVERFITTING!

Optimal degree: 2


---
## 7. Ridge Regression: Regularization

Instead of limiting the polynomial degree, we can use **regularization** to prevent overfitting.

Ridge regression adds a penalty term: minimize `||y - Xθ||² + λ||θ||²`

- **λ = 0**: No regularization (overfitting)
- **λ → ∞**: Heavy regularization (underfitting)
- **Right λ**: Prevents overfitting while keeping flexibility

**Run the cell multiple times** to cycle through λ values.

In [8]:
# Ridge Regression - Regularization Demo
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- TOGGLE LOGIC ---
try:
    lambda_idx
except NameError:
    lambda_idx = -1

lambda_values = [0, 0.01, 1, 100, 10000]
lambda_labels = ['0 (no reg)', '0.01', '1', '100', '10000 (heavy)']
lambda_idx = (lambda_idx + 1) % len(lambda_values)
current_lambda = lambda_values[lambda_idx]

# --- 1. DATA ---
# Simulated pharmacokinetic study - no real patient data is used.
np.random.seed(42)
n_points = 20
# Daily dose of an orally administered drug (5 to 25 mg/day)
dose = np.linspace(5, 25, n_points)

# Steady-state plasma concentration under SATURABLE (capacity-limited) elimination.
# Michaelis-Menten:  Css = Km * Dose / (Vmax - Dose)
#   Vmax = 30 mg/day is the most the metabolizing enzyme can clear in a day
#   Km   = 4  mg/L   is the concentration at half-maximal clearance
# Well below Vmax the enzyme has spare capacity and concentration rises almost
# in step with dose. As the dose approaches Vmax the enzyme saturates, each extra
# milligram is cleared more slowly, and concentration rises FASTER than dose.
Vmax, Km = 30.0, 4.0
true_conc = Km * dose / (Vmax - dose)
conc = true_conc + np.random.normal(0, 2.5, n_points)
conc = np.maximum(conc, 0.2)  # Concentration can't be negative

# --- 2. HIGH-DEGREE POLYNOMIAL WITH RIDGE ---
degree = 15  # High degree - would overfit without regularization

# Normalize features for stability
dose_norm = (dose - dose.mean()) / dose.std()
X_poly = np.column_stack([dose_norm**i for i in range(degree + 1)])

# Ridge regression: (X'X + λI)^-1 X'y
I = np.eye(degree + 1)
I[0, 0] = 0  # Don't regularize the bias term
theta = np.linalg.solve(X_poly.T @ X_poly + current_lambda * I, X_poly.T @ conc)

# Predictions
dose_smooth = np.linspace(5, 25, 500)
dose_smooth_norm = (dose_smooth - dose.mean()) / dose.std()
X_smooth = np.column_stack([dose_smooth_norm**i for i in range(degree + 1)])
pred_smooth = X_smooth @ theta

# Coefficient magnitude (excluding bias)
coef_magnitude = np.sum(theta[1:]**2)

# --- 3. VISUALIZE ---
fig = make_subplots(
    rows=1, cols=2,
    column_widths=[0.65, 0.35],
    subplot_titles=(
        f'Ridge Regression (λ = {lambda_labels[lambda_idx]})',
        'Coefficient Magnitudes'
    )
)

# Data
fig.add_trace(go.Scatter(
    x=dose, y=conc,
    mode='markers',
    marker=dict(size=10, color='blue'),
    name='Data'
), row=1, col=1)

# True function
fig.add_trace(go.Scatter(
    x=dose_smooth, y=Km * dose_smooth / (Vmax - dose_smooth),
    mode='lines',
    line=dict(color='green', width=2, dash='dash'),
    name='True'
), row=1, col=1)

# Ridge fit
fig.add_trace(go.Scatter(
    x=dose_smooth, y=pred_smooth,
    mode='lines',
    line=dict(color='red', width=3),
    name='Ridge fit'
), row=1, col=1)

# Coefficient magnitudes (show first 10 non-bias terms)
coef_display = np.abs(theta[1:11])
fig.add_trace(go.Bar(
    x=[f'θ{i}' for i in range(1, 11)],
    y=coef_display,
    marker_color='purple',
    name='|θ|'
), row=1, col=2)

fig.update_xaxes(title_text="Dose (mg/day)", row=1, col=1)
fig.update_yaxes(title_text="Concentration (mg/L)", range=[-10, 40], row=1, col=1)
fig.update_yaxes(title_text="|Coefficient|", row=1, col=2)

# Determine regularization status
if current_lambda == 0:
    reg_status = "NO REGULARIZATION - Overfitting!"
elif current_lambda >= 10000:
    reg_status = "HEAVY REGULARIZATION - Underfitting!"
elif current_lambda >= 1:
    reg_status = "GOOD REGULARIZATION"
else:
    reg_status = "LIGHT REGULARIZATION"

fig.update_layout(
    title=f'{reg_status} | λ={lambda_labels[lambda_idx]} | Σ|θ|²={coef_magnitude:.1f} | Run again to change λ',
    width=1000, height=450,
    showlegend=True,
    legend=dict(x=0.02, y=0.98)
)

fig.show()

print(f"\nλ = {current_lambda}")
print(f"Sum of squared coefficients: {coef_magnitude:.2f}")
if current_lambda == 0:
    print("→ Coefficients are huge, model is overfitting!")
elif current_lambda >= 10000:
    print("→ Coefficients are tiny, model is underfitting (too constrained)!")
elif current_lambda >= 1:
    print("→ Coefficients are controlled, good balance!")



λ = 0
Sum of squared coefficients: 3871827.44
→ Coefficients are huge, model is overfitting!


---
## 8. Using scikit-learn

In practice, use scikit-learn's `PolynomialFeatures` and `Ridge` classes.

This cell demonstrates the standard workflow.

In [9]:
# Polynomial Regression with scikit-learn
import numpy as np
import plotly.graph_objects as go
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

# --- DATA ---
# Simulated pharmacokinetic study - no real patient data is used.
np.random.seed(42)
n_points = 50
dose = np.linspace(5, 25, n_points).reshape(-1, 1)
# Saturable (Michaelis-Menten) elimination: Css = Km * Dose / (Vmax - Dose)
Vmax, Km = 30.0, 4.0
conc = Km * dose.flatten() / (Vmax - dose.flatten()) + np.random.normal(0, 1.2, n_points)
conc = np.maximum(conc, 0.2)

# --- CREATE PIPELINES ---
# Pipeline combines feature transformation and model fitting
linear_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=1)),
    ('model', LinearRegression())
])

poly3_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=3)),
    ('model', LinearRegression())
])

poly10_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=10)),
    ('model', LinearRegression())
])

ridge_pipe = Pipeline([
    ('poly', PolynomialFeatures(degree=10)),
    ('model', Ridge(alpha=1.0))
])

# --- FIT ALL MODELS ---
models = {
    'Linear (deg=1)': linear_pipe,
    'Poly (deg=3)': poly3_pipe,
    'Poly (deg=10)': poly10_pipe,
    'Ridge (deg=10, α=1)': ridge_pipe
}

dose_smooth = np.linspace(5, 25, 200).reshape(-1, 1)
predictions = {}
cv_scores = {}

for name, pipe in models.items():
    pipe.fit(dose, conc)
    predictions[name] = pipe.predict(dose_smooth)
    # 5-fold cross-validation
    scores = cross_val_score(pipe, dose, conc, cv=5, scoring='neg_mean_squared_error')
    cv_scores[name] = -scores.mean()

# --- VISUALIZE ---
fig = go.Figure()

# Data
fig.add_trace(go.Scatter(
    x=dose.flatten(), y=conc,
    mode='markers',
    marker=dict(size=8, color='blue', opacity=0.5),
    name='Data'
))

# True function
fig.add_trace(go.Scatter(
    x=dose_smooth.flatten(), y=Km * dose_smooth.flatten() / (Vmax - dose_smooth.flatten()),
    mode='lines',
    line=dict(color='black', width=2, dash='dash'),
    name='True (saturable PK)'
))

# Model predictions
colors = ['orange', 'green', 'red', 'purple']
for (name, pred), color in zip(predictions.items(), colors):
    fig.add_trace(go.Scatter(
        x=dose_smooth.flatten(), y=pred,
        mode='lines',
        line=dict(color=color, width=2),
        name=f'{name} (CV MSE: {cv_scores[name]:.1f})'
    ))

fig.update_layout(
    title='Polynomial Regression with scikit-learn',
    xaxis_title='Dose (mg/day)',
    yaxis_title='Plasma Concentration (mg/L)',
    width=900, height=500,
    yaxis_range=[-10, 40],
    legend=dict(x=0.02, y=0.98)
)

fig.show()

print("\nCross-Validation MSE (lower is better):")
for name, score in sorted(cv_scores.items(), key=lambda x: x[1]):
    print(f"  {name}: {score:.2f}")

best_model = min(cv_scores, key=cv_scores.get)
print(f"\nBest model by CV: {best_model}")


/Users/randygiffen/.venvs/ai-fundamentals-healthcare/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.215219700583456e-29.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/randygiffen/.venvs/ai-fundamentals-healthcare/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.3739161606504916e-29.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/randygiffen/.venvs/ai-fundamentals-healthcare/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected: slice 0 has rcond = 3.539442572889092e-29.
  return linalg.solve(A, Xy, assume_a="pos", overwrite_a=True).T
/Users/randygiffen/.venvs/ai-fundamentals-healthcare/lib/python3.12/site-packages/sklearn/linear_model/_ridge.py:227: LinAlgWarning: An ill-conditioned matrix detected:


Cross-Validation MSE (lower is better):
  Poly (deg=3): 7.49
  Linear (deg=1): 17.93
  Poly (deg=10): 53.41
  Ridge (deg=10, α=1): 53219.65

Best model by CV: Poly (deg=3)


---
## Summary

### Key Takeaways

1. **Polynomial regression** extends linear regression to capture non-linear relationships
2. **Feature transformation** is the key: transform `x` into `[1, x, x², ...]`
3. **Degree selection** is crucial:
   - Too low → underfitting (high bias)
   - Too high → overfitting (high variance)
4. **Cross-validation** helps select the right degree
5. **Regularization (Ridge)** prevents overfitting by penalizing large coefficients

### The Clinical Point

The dose-concentration curve bends upward because the enzyme that clears the drug
**saturates**. Once `Vmax` is approached, a further dose increase buys a
disproportionately large jump in plasma concentration. A linear model does not just fit
worse — it systematically under-predicts exposure at exactly the doses where toxicity
occurs. Getting the functional form right is a patient-safety question, not a
goodness-of-fit question.

Note also that the polynomial is only an **approximation** of the Michaelis-Menten
mechanism. It has no `Vmax`, so it should never be extrapolated outside the observed
dose range.

### Best Practices

- Always visualize your data first to understand the relationship
- Use cross-validation to select model complexity
- Consider Ridge regression for high-degree polynomials
- Normalize/standardize features for numerical stability
- Start simple (lower degree) and increase complexity only if needed
- Where a mechanistic model exists (as it does here), prefer it to a polynomial for extrapolation
